# llamacpp-chat2（Google Colab）セットアップ

この Notebook は **llamacpp-chat2** の Colab GPU 向け手順です。

1. Google Drive をマウントし、リポジトリを配置
2. `llama-server` をビルド（CUDA。Drive 上にキャッシュ）
3. 一時領域 `/content` に venv を作成し、制御API + Gradio を起動

| 用途 | パス | 寿命 |
|------|------|------|
| リポジトリ | `/content/drive/MyDrive/llamacpp-chat2` | Drive 永続 |
| llama.cpp | `.../vendor/llama.cpp` | Drive 永続 |
| venv / モデル | `/content/llamacpp-chat2` | ランタイム再起動で消失 |

## 事前準備

**ランタイム → ランタイムのタイプを変更 → ハードウェア アクセラレータ → GPU**

Public URL（Gradio share）が表示されたら手元ブラウザで開いてください。


## GPU 確認

In [ ]:
!nvidia-smi

## Drive マウントとパス設定

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/mckey-dev/llamacpp-chat2.git"
REPO = Path("/content/drive/MyDrive/llamacpp-chat2")
RUNTIME = Path("/content/llamacpp-chat2")
MODELS_DIR = RUNTIME
VENV_SERVER = RUNTIME / ".venv-server"
VENV_FRONTEND = RUNTIME / ".venv-frontend"
PY_SERVER = VENV_SERVER / "bin" / "python"
PY_FRONTEND = VENV_FRONTEND / "bin" / "python"
CONTROL_TOKEN = "llamacpp-chat2"

RUNTIME.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("REPO (永続):", REPO)
print("RUNTIME (一時):", RUNTIME)
print("MODELS_DIR:", MODELS_DIR)

## リポジトリクローン / 更新

In [ ]:
import subprocess

if not REPO.is_dir():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    print("クローンしました:", REPO)
else:
    print("リポジトリは既に存在します:", REPO)
    try:
        subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only"])
    except subprocess.CalledProcessError:
        print("git pull に失敗しました。手動で更新してください。")

for name in ("server/control_api.py", "ui/app.py", "requirements-frontend.txt"):
    assert (REPO / name).is_file(), f"必須ファイルがありません: {REPO / name}"
print("リポジトリ OK")

## ビルドツールと llama-server

In [ ]:

import subprocess

# Colab 用（初回のみ）
subprocess.run(["apt-get", "update"], check=False)
subprocess.run(["apt-get", "install", "-y", "cmake", "build-essential"], check=False)

LLAMA_CPP_DIR = REPO / "vendor" / "llama.cpp"
LLAMA_SERVER = LLAMA_CPP_DIR / "build" / "bin" / "llama-server"

if not LLAMA_SERVER.is_file():
    LLAMA_CPP_DIR.parent.mkdir(parents=True, exist_ok=True)
    if not LLAMA_CPP_DIR.is_dir():
        print("llama.cpp をクローンします…")
        subprocess.check_call(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/ggml-org/llama.cpp.git",
                str(LLAMA_CPP_DIR),
            ]
        )
    print("llama-server をビルドします（CUDA）。初回は十数分かかることがあります…")
    subprocess.check_call(
        ["cmake", "-B", "build", "-DGGML_CUDA=ON"],
        cwd=str(LLAMA_CPP_DIR),
    )
    subprocess.check_call(
        ["cmake", "--build", "build", "--config", "Release", "-j"],
        cwd=str(LLAMA_CPP_DIR),
    )

assert LLAMA_SERVER.is_file(), f"llama-server がありません: {LLAMA_SERVER}"
print("llama-server:", LLAMA_SERVER)

## Python venv とフロント依存

In [ ]:
def _stream(cmd, *, cwd=None, env=None):
    """サブプロセスを実行し、出力を Notebook に逐次表示する。"""
    import subprocess

    print("$", " ".join(str(c) for c in cmd))
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"終了コード {code}: {' '.join(str(c) for c in cmd)}")

import subprocess
import sys

if not PY_SERVER.is_file():
    print("Creating .venv-server …")
    subprocess.check_call([sys.executable, "-m", "venv", str(VENV_SERVER)])

if not PY_FRONTEND.is_file():
    print("Creating .venv-frontend …")
    subprocess.check_call([sys.executable, "-m", "venv", str(VENV_FRONTEND)])

_stream(
    [str(PY_FRONTEND), "-m", "pip", "install", "--upgrade", "pip"],
    cwd=str(REPO),
)
_stream(
    [
        str(PY_FRONTEND),
        "-m",
        "pip",
        "install",
        "-r",
        str(REPO / "requirements-frontend.txt"),
    ],
    cwd=str(REPO),
)
print("venv-server:", PY_SERVER)
print("venv-frontend:", PY_FRONTEND)

## 環境変数

In [ ]:
import os

MODELS_DIR.mkdir(parents=True, exist_ok=True)

os.environ["LLAMACPP_CHAT2_ROOT"] = str(REPO)
os.environ["LLAMACPP_CHAT2_MODELS_DIR"] = str(MODELS_DIR)
os.environ["LLAMACPP_CHAT2_LLAMA_SERVER"] = str(LLAMA_SERVER)
os.environ["LLAMACPP_CHAT2_CONTROL_TOKEN"] = CONTROL_TOKEN
os.environ["LLAMACPP_CHAT2_LLAMA_HOST"] = "127.0.0.1"
os.environ["LLAMACPP_CHAT2_LLAMA_PORT"] = "8080"
os.environ["LLAMACPP_CHAT2_CONTROL_HOST"] = "127.0.0.1"
os.environ["LLAMACPP_CHAT2_CONTROL_PORT"] = "8090"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["GRADIO_SHARE"] = "True"
os.environ["GRADIO_SERVER_NAME"] = "0.0.0.0"

print("MODELS_DIR:", MODELS_DIR)
print("LLAMA_SERVER:", LLAMA_SERVER)
print("制御API: http://127.0.0.1:8090")
print("推論:    http://127.0.0.1:8080")

## 制御API 起動（バックグラウンド）

In [ ]:
import os
import socket
import subprocess
import time

if "CONTROL_PROC" in globals() and CONTROL_PROC and CONTROL_PROC.poll() is None:
    print("制御APIは既に起動しています (pid=", CONTROL_PROC.pid, ")")
else:
    env = os.environ.copy()
    CONTROL_PROC = subprocess.Popen(
        [str(PY_SERVER), "-m", "server.control_api"],
        cwd=str(REPO),
        env=env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
    )
    print("制御APIを起動しました pid=", CONTROL_PROC.pid)

    ok = False
    for _ in range(60):
        if CONTROL_PROC.poll() is not None:
            raise RuntimeError(
                f"制御APIが終了しました exit_code={CONTROL_PROC.returncode}"
            )
        try:
            with socket.create_connection(("127.0.0.1", 8090), timeout=1):
                ok = True
                break
        except OSError:
            time.sleep(0.5)
    if not ok:
        raise RuntimeError("制御API (8090) が応答しません")
    print("制御API ready: http://127.0.0.1:8090")

## Gradio UI 起動

**Public URL** が表示されたら開いてください。

ランタイム再起動後は venv 作成以降のセルを再実行してください（Drive 上のリポジトリと llama.cpp ビルドは残ります）。

停止: **ランタイム → 実行を中断**

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

_DROP_PREFIXES = (
    "clip_model_loader: tensor[",
    "add_text:",
    "add_media:",
    "tokenize:",
    "find_slot:",
    "image_tokens->",
    "batch_f32",
    "load_hparams:",
)
_KEEP_SNIPPETS = (
    "Running on",
    "public URL",
    "Public URL",
    "Gradio",
    "Error",
    "Traceback",
    "Exception",
    "エラー",
    "control API",
)


def _notebook_should_print(line: str) -> bool:
    s = line.strip()
    if not s:
        return False
    if any(k in s for k in _KEEP_SNIPPETS):
        return True
    if any(s.startswith(p) for p in _DROP_PREFIXES):
        return False
    return True

os.chdir(REPO)
env = os.environ.copy()


print("Gradio UI を起動します…")
print("Connection タブの URL は既定のまま (8080 / 8090) で動作します。")
print("1. Connection: モデル追加 → 一覧更新 → Load")
print("2. Chat: 会話（画像添付可）")
print("停止: このセルを Interrupt\n")

proc = subprocess.Popen(
    [str(PY_FRONTEND), "-m", "ui.app"],
    cwd=str(REPO),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
try:
    assert proc.stdout is not None
    for line in proc.stdout:
        if _notebook_should_print(line):
            print(line, end="")
            sys.stdout.flush()
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"ui.app が終了コード {code} で終了しました")
except KeyboardInterrupt:
    proc.terminate()
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        proc.kill()
    if "CONTROL_PROC" in globals() and CONTROL_PROC:
        CONTROL_PROC.terminate()
    print("\n停止しました")

## 仮想環境の削除

依存の入れ直しや破損した venv を消したいときに使います。**パス設定セルを先に実行**してください。

- `.venv-server` … 制御API用（標準ライブラリのみ）
- `.venv-frontend` … Gradio UI 用

削除後は「Python venv とフロント依存」セルを再実行してください。制御APIが動いている場合、server 側削除時に停止します。

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

try:
    import ipywidgets as widgets
    from IPython.display import clear_output, display
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "ipywidgets"]
    )
    import ipywidgets as widgets
    from IPython.display import clear_output, display

assert "VENV_SERVER" in globals() and "VENV_FRONTEND" in globals(), (
    "先にパス設定セルを実行してください（VENV_SERVER / VENV_FRONTEND）"
)


def _stop_control_api() -> None:
    """制御API 子プロセスがあれば停止する。"""
    proc = globals().get("CONTROL_PROC")
    if proc is None:
        return
    if proc.poll() is not None:
        return
    proc.terminate()
    try:
        proc.wait(timeout=10)
    except Exception:
        proc.kill()
    print("制御APIを停止しました")


def _rm_venv(path: Path, label: str) -> None:
    """仮想環境ディレクトリを削除する。"""
    if not path.exists():
        print(f"{label}: 存在しません（{path}）")
        return
    shutil.rmtree(path)
    print(f"{label}: 削除しました → {path}")


out = widgets.Output()


def on_delete_server(_btn) -> None:
    with out:
        clear_output()
        _stop_control_api()
        _rm_venv(VENV_SERVER, ".venv-server")


def on_delete_frontend(_btn) -> None:
    with out:
        clear_output()
        _rm_venv(VENV_FRONTEND, ".venv-frontend")


def on_delete_both(_btn) -> None:
    with out:
        clear_output()
        _stop_control_api()
        _rm_venv(VENV_SERVER, ".venv-server")
        _rm_venv(VENV_FRONTEND, ".venv-frontend")


btn_server = widgets.Button(
    description=".venv-server を削除",
    button_style="warning",
    layout=widgets.Layout(width="auto"),
)
btn_frontend = widgets.Button(
    description=".venv-frontend を削除",
    button_style="warning",
    layout=widgets.Layout(width="auto"),
)
btn_both = widgets.Button(
    description="両方削除",
    button_style="danger",
    layout=widgets.Layout(width="auto"),
)
btn_server.on_click(on_delete_server)
btn_frontend.on_click(on_delete_frontend)
btn_both.on_click(on_delete_both)

print("対象:")
print(f"  .venv-server   : {VENV_SERVER}  存在={VENV_SERVER.is_dir()}")
print(f"  .venv-frontend : {VENV_FRONTEND}  存在={VENV_FRONTEND.is_dir()}")
display(widgets.HBox([btn_server, btn_frontend, btn_both]), out)